In [ ]:
import casadi as cs
import numpy as np
from scipy.optimize import brentq
from time import perf_counter
from matplotlib import pyplot as plt
import pickle
import ctypes
import os

In [2]:
data = pickle.load(open('../data/dataset_2/dataset_21.pkl','rb'))
K = data["K"]
W = data["W"]
Nf = data["Nf"]
Ns = data["Ns"]
locomotion_NMM = np.array(data["locomotion"][2])
switching_NMM = np.array(data["switch"][2])
Emin = max(min(locomotion_NMM[:,-1]),min(switching_NMM[:,-1]))
Emax = min(max(locomotion_NMM[:,-1]),max(switching_NMM[:,-1]))

In [3]:
class Layout:

    def __init__(self, modes, N, dim):
        self.modes = tuple(modes)
        self.N = N
        self.dim = dim
        self.dim_terminal = dim[modes[-1]] - int(modes[-1] == "f")
        self.dim_terminal_aux = 2 + 2*self.dim_terminal # E, phi, s+, s-
        # stages = (mode_index, mode_name, node_index)
        self.stages = [(i,m,k) for i,m in enumerate(modes) for k in range(N[m])] 
        self.N_ocp = len(self.stages) - 1
        self.edge = []      # edge[k] = "flow"/"reset"/"terminal" 
        self.nx = []        # nx[k] = dimension of state at stage k
        self.nu = []        # nu[k] = dimension of control at stage k

        for k, (i, m, _) in enumerate(self.stages):
            final = k == self.N_ocp
            # x_aug = [x tau t]^T if not terminal 
            # x_aug = [x tau t E phi s+ s-] if terminal
            self.nx.append(dim[m] + 2 + final*self.dim_terminal_aux)
            if final:
                self.edge.append("terminal")
                self.nu.append(0)
            elif self.stages[k+1][0] != i:
                self.edge.append("reset")
                self.nu.append(1) # the control is the next mode duration
            else:
                self.edge.append("flow")
                # control of size 1 except just before the terminal state: 
                # v_(N+1) = [u_(N-1) E phi s+ s-]
                self.nu.append(1+(k==self.N_ocp-1) * self.dim_terminal_aux)

        offset = 0 # current position in the flattened QP vector
        self.x_slice, self.u_slice = [], []
        for nx,nu in zip(self.nx, self.nu):
            self.x_slice.append(slice(offset, offset+nx))
            offset += nx
            self.u_slice.append(slice(offset, offset+nu))
            offset += nu
        self.nw = offset # size of w = [x0,u0,x1,u1 ... xN]
        # find [E, phi, s_plus, s_minus] in terminal x and u 
        self.x_t_slice = slice(self.x_slice[-1].stop - self.dim_terminal_aux, self.x_slice[-1].stop)
        self.u_t_slice = slice(self.u_slice[-2].start+1, self.u_slice[-2].stop)

    def split(self,w):
        states = [w[s] for s in self.x_slice]
        controls = [w[s] for s in self.u_slice]
        return states, controls

    def state_parts(self, states, k):
        # returns (x,tau,t)
        d = self.dim[self.stages[k][1]]
        return states[k][:d], states[k][d], states[k][d+1]

    def pack(self, x, u, duration, E, phi): 
        w = []
        mode_start = [cs.sum1(duration[:i]) for i in range(len(self.modes))]
        x_t_aux = cs.vertcat(E, phi, cs.DM.zeros(2*self.dim_terminal))
        for k,(i,_,j) in enumerate(self.stages):
            # compute w_k = [... y_k v_k ...]
            y_k = cs.vertcat(x[i][:, j], duration[i], mode_start[i])
            w.append(cs.vertcat(y_k, x_t_aux) if k == self.N_ocp else y_k)
            if self.edge[k] == "flow":
                # v_k = u_ij
                w.append(cs.vertcat(u[i][j], x_t_aux) if k == self.N_ocp-1 else u[i][j])
            elif self.edge[k] == "reset":
                # v_k = tau_(i+1)
                w.append(duration[i+1])
        return cs.vertcat(*w)



In [4]:
def bounded(expression, lower=0, upper=0):
        # converts expression in three equally sized vectors inf < expression < upper 
        # by default, expression = 0
        expression = cs.vec(expression)
        ones = cs.DM.ones(expression.numel())
        return expression, lower*ones, upper*ones

In [5]:
def RK4(f,x,h,u):
    k1 = f(x,u)
    k2 = f(x+0.5*h*k1,u)
    k3 = f(x+0.5*h*k2,u)
    k4 = f(x+h*k3,u)
    x_next = x + h/6 * (k1 + 2*k2 + 2*k3 +k4)
    return x_next

In [6]:
xs = cs.SX.sym("xs", 4)
xf = cs.SX.sym("xf", 6)
u = cs.SX.sym("u")
h = cs.SX.sym("h")

l = cs.sqrt(xs[0] ** 2 + xs[1] ** 2)
dxs = cs.vertcat(
    xs[2:], 
    (K * (1 - l) + u) * xs[0] / l, 
    (K * (1 - l) + u) * xs[1] / l - 1
)
dxf = cs.vertcat(
    xf[3:], 
    0, 
    -1, 
    -(W**2) * xf[2] + u
)
xfs = cs.vertcat(
    xs[:2],
    cs.atan2(-xs[0],xs[1]),
    xs[2:],
    (xs[0] * xs[3] - xs[1] * xs[2]) / l**2
)
xsf = cs.vertcat(
    -cs.sin(xf[2]),
    xf[[1,3,4]]
)
e_TD = cs.cos(xf[2]) - xf[1]
e_LO = l - 1
de_TD = -cs.sin(xf[2])*xf[5] - xf[4]
de_LO = xs[0]*xs[2] + xs[1]*xs[3]


In [7]:
f =  {
    "f": cs.Function("ff", [xf, u], [dxf]), 
    "s": cs.Function("fs", [xs, u], [dxs])
}
rk4 = {
    "f": cs.Function("RK4f", [xf,h,u],[RK4(f["f"],xf,h,u)]),
    "s": cs.Function("RK4s", [xs,h,u],[RK4(f["s"],xs,h,u)])
}
convert = {
    "s": cs.Function("to_flight", [xs],[xfs]),
    "f": cs.Function("to_stance", [xf],[xsf])
}
guard = {
    "s" : cs.Function("guards", [xs], [e_LO]),
    "f" : cs.Function("guardf", [xf], [e_TD])
}
guard_rate = {
    "s" : cs.Function("guards", [xs], [de_LO]),
    "f" : cs.Function("guardf", [xf], [de_TD])
}
next_mode = {
    "f":"s", 
    "s":"f"
}
dim = {
    "f":6, 
    "s":4
}
N = {
    "f":15, 
    "s":20
}

In [8]:
def integrate_until_event(x0,mode,N_max=1000,h=1e-2):
    x = x0
    t = 0.0
    xh = []
    th = []
    armed = False
    
    for i in range(N_max):

        xh.append(x)
        th.append(t)
        x_next = rk4[mode](x,h,0.0)

        g0 = guard[mode](x)
        g1 = guard[mode](x_next)

        if (not armed) and g1 < 0 and g0<0:
            armed = True

        if g0 * g1 < 0 and armed:
            def residual(tau):
                x_tau = rk4[mode](x,tau,0.0)
                return guard[mode](x_tau)
            
            tau_event = brentq(residual,0.0,h,xtol=1e-6,rtol=1e-6)
            x_event = rk4[mode](x,tau_event,0.0)
            xh.append(x_event)
            th.append(t+tau_event)

            return np.column_stack(th),np.column_stack(xh)

        t += h
        x = x_next

In [9]:
def energy_sample(M,grid):
    idx = np.argsort(M[:,-1])
    M = M[idx,:]
    E = M[:,-1]
    return np.column_stack([np.interp(grid, E, M[:, c]) for c in range(M.shape[1])])
        
def trajectory_sample(u,grid):
    x0 = u[:6]
    th,xh = integrate_until_event(x0,"f")
    tTD,xTD = th[0,-1],convert["f"](xh[:,-1])
    th_s,xh_s = integrate_until_event(xTD,"s")
    tLO,xLO = th_s[0,-1],convert["s"](xh_s[:,-1])
    th_f,xh_f = integrate_until_event(xLO,"f")
    ts_norm = th_s / th_s[0,-1]
    tf_norm = th_f / th_f[0,-1] 

    traj_s = np.column_stack([
        np.interp(grid, ts_norm.squeeze(), xh_s[r,:]) for r in range(xh_s.shape[0])
    ]+[np.interp(grid, ts_norm.squeeze(), th_s.squeeze())])
    traj_f = np.column_stack([
        np.interp(grid, tf_norm.squeeze(), xh_f[r,:]) for r in range(xh_f.shape[0])
    ]+[np.interp(grid, tf_norm.squeeze(), th_f.squeeze())])
    # traj_s = convert["s"](traj_s.T).T

    return traj_s,traj_f

def surface(M,grid):
    N = M.shape[0]
    surf_s,surf_f = np.zeros((N,N,5)),np.zeros((N,N,7))
    for i in range(N):
        ts,tf = trajectory_sample(M[i],grid)
        surf_s[:,i,:] = ts
        surf_f[:,i,:] = tf
    return surf_s,surf_f

In [10]:
def make_problem(layout,target_LUT,rho_regul=1e-6,rho_slack=1e3):
    w = cs.MX.sym("w", layout.nw)
    p = cs.MX.sym("p", layout.dim[layout.modes[0]]+1)
    y,v = layout.split(w)
    x0,T = p[:-1],p[-1]

    effort = []
    for k, (_, mode, _) in enumerate(layout.stages):
        if layout.edge[k] == "flow":
            _, tau, _ = layout.state_parts(y, k)
            h = tau/(layout.N[mode]-1)
            effort.append(cs.sqrt(2*h)*v[k][0])

    E,phi,s = cs.vertsplit(y[-1][-layout.dim_terminal_aux:], [0,1,2,layout.dim_terminal_aux])
    s_plus, s_minus = cs.vertsplit(s, [0, layout.dim_terminal, 2*layout.dim_terminal])
    q = cs.vertcat(*effort)
    cost = 0.5*cs.dot(q,q) + rho_slack * cs.sum1(s)
    grad = cs.gradient(cost, w)
    Q = cs.jacobian(q, w)
    H = Q.T @ Q + rho_regul * cs.DM.eye(layout.nw)

    residuals = []          # all constraint expressions OCP-ordered: [dynamics gap_k \\ local constraints_k]
    lower,upper = [],[]     # corresponding bounds : lower <= residuals <= upper
    ng = []                 # ng[k]: number non-dynamics constraint at stage k
    for k, (idx_m,m,idx_n) in enumerate(layout.stages):
        x,tau,t = layout.state_parts(y,k)


        if k<layout.N_ocp: # non terminal stage
            if layout.edge[k] == "flow":
                h = tau / (layout.N[m] - 1)
                y_est = cs.vertcat(rk4[m](x,h,v[k][0]),tau,t)
                if k == layout.N_ocp-1:
                    # propagate the augmented capture state using augmented control v_(N-1)
                    y_est = cs.vertcat(y_est, v[k][1:])
            else: # reset stage
                y_est = cs.vertcat(convert[m](x),v[k][0], tau+t)
            gap,lb,ub = bounded(y_est-y[k+1])
            residuals.append(gap)
            lower.append(lb)
            upper.append(ub)

        local = [] # local non-dynamics constraints
        if k == 0:
            local.append(bounded(cs.vertcat(x-x0,t)))
        if layout.edge[k] == "reset":
            local.append(bounded(guard[m](x)))
            local.append(bounded(guard_rate[m](x),1e-3, cs.inf))
        if idx_m == len(layout.modes)-1 and idx_n > 0:
            # every nodes of the last mode except the first one
            local.append(bounded(guard[m](x), -cs.inf, -1e-3))
        if k == layout.N_ocp: # terminal 
            target = target_LUT(cs.vertcat(E,phi))
            x_target = target[1:-1] if m == "f" else target[:-1]
            x_capture = x[1:] if m == "f" else x
            local.append(bounded(x_target-x_capture-s_plus+s_minus))
            local.append(bounded(t+tau-T))

        ng.append(sum(expression.numel() for expression, _, _ in local))
        for expression, lb, ub in local:
            residuals.append(expression)
            lower.append(lb)
            upper.append(ub)

    residual = cs.vertcat(*residuals)
    A = cs.jacobian(residual, w)
    linearize = cs.Function(
        "linearize",
        [w,p],
        [grad,H,A,residual],
        {"jit": True, "compiler": "shell"}
    )
    values = cs.Function(
        "values",
        [w,p],
        [cost,residual]
    )
    structure = {"h": H.sparsity(), "a": A.sparsity()}
    return linearize, values, structure, cs.vertcat(*lower), cs.vertcat(*upper), ng





In [11]:
def make_bounds(layout, x_bounds, u_bounds, t_bounds, E_bounds, phi_bounds):
    lower = np.full(layout.nw, -np.inf)
    upper = np.full(layout.nw, np.inf)
    for k, (idx_m, m, _) in enumerate(layout.stages):
        d = layout.dim[m]
        y_s,v_s = layout.x_slice[k], layout.u_slice[k]
        for idx_b,bounds in x_bounds.get(m, {}).items():
            lower[y_s.start+idx_b], upper[y_s.start+idx_b] = bounds 
        lower[y_s.start+d],upper[y_s.start+d] = t_bounds[m]
        if layout.edge[k] == "flow":
            lower[v_s.start], upper[v_s.start] = u_bounds[m]
        elif layout.edge[k] == "reset":
            # tmin < t(k+1) < tmax
            lower[v_s], upper[v_s] = t_bounds[layout.modes[idx_m + 1]]

    for aux in (layout.u_t_slice,layout.x_t_slice):
        lower[aux.start],upper[aux.start] = E_bounds
        lower[aux.start+1],upper[aux.start+1] = phi_bounds
        lower[aux.start+2:aux.stop] = 0.0

    return lower,upper

def make_delta(layout):
    # constructs the trust-region radius for every entry in delta w
    delta = np.full(layout.nw, np.inf)
    for k, (_,m,_) in enumerate(layout.stages):
        d = layout.dim[m]
        y_s,v_s = layout.x_slice[k], layout.u_slice[k]
        delta[y_s.start:y_s.start + d] = 0.2    # physical state increment bound
        delta[y_s.start+d] = 0.1                # mode duration increment bound
        if layout.edge[k] == "flow":
            delta[v_s.start] = 0.5              # control signal increment bound
        elif layout.edge[k] == "reset":
            delta[v_s] = 0.1                    # mode duration increment bound
    for aux in (layout.u_t_slice,layout.x_t_slice):
        delta[aux.start] = 0.1                  # E increment bound
        delta[aux.start+1] = 0.2                # phi increment bound
    return delta

In [12]:
def qp_data(w,p,linearize,con_min,con_max,w_min,w_max,delta):
    grad,H,A,residual = linearize(w,p)
    nominal = np.asarray(w).squeeze()
    return {
        "h": H, "g": grad, "a": A,
        "lbx": np.maximum(w_min-nominal, -delta),
        "ubx": np.minimum(w_max-nominal, delta),
        "lba": con_min-residual, "uba": con_max-residual,
    }

In [13]:
def initialize_mode(LUT, count, E0, phi_end=1.0):
    return np.column_stack([
        np.asarray(LUT([E0, phi])[:-1]).squeeze()
        for phi in np.linspace(0, phi_end, count)
    ])

In [14]:
libc = ctypes.CDLL(None)
def solve_qp(qp,solver):
    # wrapper to remove unwanted output from hpipm
    stdout, null = os.dup(1), os.open(os.devnull, os.O_WRONLY)
    try:  # CasADi 3.7.2's HPIPM plugin prints directly to C stdout
        os.dup2(null, 1)
        return solver(**qp)
    finally:
        libc.fflush(None)
        os.dup2(stdout, 1)
        os.close(stdout)
        os.close(null)

In [15]:
def evaluate(w,p,values,r_min,r_max,mu=1e3):
    J,r = values(w,p)
    violation = np.maximum(r_min-r,0) + np.maximum(r-r_max,0)
    merit = J + mu * np.sum(violation)
    return merit

In [16]:
def line_search(w_bar,p_bar,delta_w,values,g_min,g_max,mu=1e3):
    merit = evaluate(w_bar,p_bar,values,g_min,g_max,mu)
    alpha = 1.0
    while alpha > 1e-6:
        trial = w_bar + alpha*delta_w 
        if evaluate(trial,p_bar,values,g_min,g_max,mu) < merit:
            return False, alpha
        alpha *= 0.5
    return True, alpha

In [ ]:
def alternating(mode,n):
    return tuple(mode if k%2 == 0 else next_mode[mode] for k in range(n))

def candidate_sequences(mode, nominal, max_modes=4):
    counts = {
        "nominal": nominal, "minus": nominal - 1, "plus": nominal + 1,
    }
    return {
        name: alternating(mode, n) for name, n in counts.items() if 1 <= n <= max_modes
    }

def select_candidate(candidates):
    accepted = [candidate for candidate in candidates if candidate["success"]]
    return min(accepted, key=lambda c: c["merit"]) if accepted else None

In [ ]:
def RTI_candidate(solver, seed, x0, T, mu):
    p = cs.vertcat(x0,T)
    w_bar = solver.layout.pack(seed["x"],seed["u"],seed["t"],seed["E"],seed["phi"])
    qp = qp_data(
        w_bar, p, solver.linearize,
        solver.con_min, solver.con_max,
        solver.w_min, solver.w_max, solver.delta
    )
    qp["x0"] = cs.DM.zeros(solver.layout.nw) # useless ?

    result = solve_qp(qp,solver.qp_solver)
    if not solver.qp_solver.stats()["success"]:
        return {"success" : False}

    w = w_bar + result["x"] # alpha = 1
    cost,residual = solver.values(w,p)

    violation = np.maximum(solver.con_min - residual,0) + np.maximum(residual - solver.con_max,0)
    violation_sum = float(cs.sum1(violation))
    return {
        "success": True,
        "w": w,
        "trajectory": unpack_trajectory(solver.layout,w), ## TODO
        "cost": float(cost),
        "violation": float(cs.mmax(violation)),
        "merit" : float(cost) + mu * violation_sum
    }

In [ ]:
E_grid = np.linspace(Emin, Emax, 20)
locomotion_NMM_sampled = energy_sample(locomotion_NMM,E_grid)
switching_NMM_sampled = energy_sample(switching_NMM,E_grid)

phi_grid = np.linspace(0,1,20)
ss_loc,sf_loc = surface(locomotion_NMM_sampled,phi_grid)
ss_swi,sf_swi = surface(switching_NMM_sampled,phi_grid)

values_sloc = ss_loc.ravel(order="C")
values_floc = sf_loc.ravel(order="C")
values_sswi = ss_swi.ravel(order="C")
values_fswi = sf_swi.ravel(order="C")
LUT_sloc = cs.interpolant('LUT_sloc', 'bspline',[E_grid,phi_grid],values_sloc)
LUT_floc = cs.interpolant('LUT_floc', 'bspline',[E_grid,phi_grid],values_floc)
LUT_sswi = cs.interpolant('LUT_sswi', 'bspline',[E_grid,phi_grid],values_sswi)
LUT_fswi = cs.interpolant('LUT_fswi', 'bspline',[E_grid,phi_grid],values_fswi)

In [ ]:
E0 = 4
E_inf = max(Emin,E0-0.2)
E_sup = min(Emax,E0+0.2)

x_bounds = {
    "f" : {1: (0.0, np.inf)}, 
    "s" : {}
}
u_bounds = {
    "f" : (-1,1), 
    "s" : (-1,1)
}
t_bounds = {
    "f": (1e-2, 10.0),
    "s": (1e-2, 10.0),
}
E_bounds = (E_inf,E_sup)
phi_bounds = (0.01,0.99)

bounds = (x_bounds, u_bounds, t_bounds, E_bounds, phi_bounds)

In [ ]:
sequence_variants = [alternating(m,n) for m in 'fs' for n in range(1,5)]
targets = {
    'locomotion': {'f':LUT_floc,'s':LUT_sloc},
    'switching': {'f':LUT_fswi,'s':LUT_sswi}
}
solvers = {
    (name,seq): Solver(LUT[seq[-1]],seq,N,dim,bounds)
    for name,LUT in targets.items() for seq in sequence_variants
}

In [ ]:
modes = ("s", "f", "s")
layout = Layout(modes, N, dim)
linearize,values, qp_structure, con_min, con_max, ng = make_problem(layout, LUT_sswi)

qp_solver = cs.conic(
    "qp_solver",
    "hpipm",
    qp_structure,
    {
        "N": layout.N_ocp, 
        "nx": layout.nx, 
        "nu": layout.nu, 
        "ng": ng,
        "print_time": False, 
        "verbose": False, 
        "error_on_fail": False,
        "hpipm": {
            "mode": "balance",
            "warm_start": 1,
        },
    },
)

In [ ]:

ts,tf = LUT_sloc([E0,1])[-1],LUT_floc([E0,1])[-1]
T = float(ts + tf + 0.5*ts)
t_init = cs.vertcat(ts,tf,0.5*ts)
xs_init = initialize_mode(LUT_sloc,N["s"],E0) 
xf_init = initialize_mode(LUT_floc,N["f"],E0)
x_init = [xs_init, xf_init, initialize_mode(LUT_sloc,N["s"],E0,0.5)]
u_init = [np.zeros(N[m]-1) for m in modes]
E_init = E0
phi_init = 0.5

p = cs.vertcat(xs_init[:,0],T)

w_min, w_max = make_bounds(layout, x_bounds, u_bounds, t_bounds, E_bounds, phi_bounds)
delta = make_delta(layout)

In [ ]:
successful = initialization
elapsed = 0.0 
Ts = 5e-2
mu = 2000

controller_running = True
while controller_running:

    x0,m = measure_state()

    shifted,nominal = shift_trajectory(successful, x0, m, elapsed)

    sequences = candidate_sequences(m, nominal)

    seeds = {}
    if "nominal" in sequences:
        seeds["nominal"] = warm_candidate(shifted, sequences["nominal"],T)
    if "minus" in sequences:
        seeds["minus"] = warm_candidate(shifted, sequences["minus"],T)
    if "plus" in sequences:
        seeds["plus"] = passive_candidate(shifted, sequences["plus"],T)

    candidates = []
    for name,seq in sequences.items():
        solver = solvers[name, seq]
        candidate = RTI_candidate(solver, seeds[name], x0, T, mu)
        candidate["name"] = name
        candidate["sequence"] = seq
        candidates.append(candidate)

    selected = select_candidate(candidates)
    if selected is not None:
        u = selected["trajectory"]["u"][0][0]
        successful = selected["trajectory"]
        elapsed = Ts 
    else: 
        u = shifted["u"][0][0]
        elapsed += Ts 

    apply_control(u, Ts)

NameError: name 'initialization' is not defined

In [ ]:
max_iter = 50
tolerance = 1e-6

history = []
success = False 
w_bar = layout.pack(x_init,u_init,t_init,E_init,phi_init)
x0 = cs.DM.zeros(layout.nw)
start = perf_counter()
for iteration in range(max_iter):
    qp = qp_data(w_bar, p, linearize, con_min, con_max, w_min, w_max, delta)
    qp["x0"] = x0
    result = solve_qp(qp,qp_solver)
    if not qp_solver.stats()["success"]:
        print("Failed ", iteration)
        break
    
    delta_w = result["x"]
    lambda_a = result["lam_a"]
    lambda_x = result["lam_x"]
    mu = max(2000, 1.1*np.max(np.abs(lambda_a)))

    no_improvement, alpha = line_search(w_bar,p,delta_w,values,con_min,con_max,mu)
    w_bar = w_bar + alpha * delta_w 
    _,r = values(w_bar,p)
    max_violation = np.max(np.maximum(con_min-r, 0) + np.maximum(r-con_max, 0))
    success = max_violation < tolerance

    x0 = (1-alpha) * delta_w

    if success:
        print("Success" , iteration)
        break
print(perf_counter() - start)


Success 6
0.07046163799986971
